# EUR/USD Time Series Project — Lesson 2
## Phase 1 — Lesson 2: Time Index, Frequency, and Sampling

This notebook applies Lesson 2 to `DATA/EURUSD240.csv`.

### Goals
- Load the headerless EUR/USD file
- Create a `DateTimeIndex`
- Confirm chronological order
- Understand time index vs frequency
- Detect the normal sampling interval
- Confirm the data is H4 / 4-hour data
- Understand weekend/holiday gaps
- Select observations by date and time
- Count candles per day
- Keep the original H4 data unchanged


## 1. Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load the Dataset

The file column order is:

`Date, Time, Open, High, Low, Close, Volume`


In [ ]:
columns = ["Date", "Time", "Open", "High", "Low", "Close", "Volume"]

eurusd = pd.read_csv(
    "DATA/EURUSD240.csv",
    header=None,
    names=columns
)

eurusd.head()

## 3. Create the DateTime Index

In [ ]:
eurusd["DateTime"] = pd.to_datetime(
    eurusd["Date"] + " " + eurusd["Time"],
    format="%Y.%m.%d %H:%M"
)

eurusd = eurusd.set_index("DateTime")
eurusd = eurusd.drop(columns=["Date", "Time"])
eurusd = eurusd.sort_index()

eurusd.head()

## 4. Inspect the Time Index

In [ ]:
print(type(eurusd.index))
print("First timestamp:", eurusd.index.min())
print("Last timestamp :", eurusd.index.max())
print("Rows           :", len(eurusd))

### Is the index sorted?

For time series analysis, we normally want:

**oldest → newest**


In [ ]:
eurusd.index.is_monotonic_increasing

## 5. Check for Duplicate Timestamps

In [ ]:
eurusd.index.duplicated().sum()

# 6. What Is Frequency?

Frequency tells us how often observations normally occur.

Examples:
- Daily → every day
- Hourly → every hour
- H4 → every 4 hours
- Weekly → every week

Your EUR/USD file is expected to be H4.


## 7. Calculate Differences Between Consecutive Timestamps

In [ ]:
time_differences = eurusd.index.to_series().diff()
time_differences.value_counts().head(10)

The most common result should be:

`0 days 04:00:00`

Therefore:

$$
4 \text{ hours} = 240 \text{ minutes}
$$

and the normal sampling interval is:

$$
\boxed{H4}
$$


## 8. Find the Most Common Sampling Interval

In [ ]:
base_interval = time_differences.mode()[0]
base_interval

Expected result:

`Timedelta('0 days 04:00:00')`

This confirms the dataset is EUR/USD H4 data.


## 9. Try Pandas Frequency Inference

In [ ]:
pd.infer_freq(eurusd.index)

`pd.infer_freq()` may return `None`.

That does not mean the data is not H4. Forex markets close during weekends and some holidays, so the calendar timestamps are not perfectly regular.

For financial data, inspecting timestamp differences is often more informative.


# 10. Weekend Gaps

You may see gaps such as:

`2 days 04:00:00`

A simplified example:

Friday evening → market closed → Monday

So distinguish between:
- **Trading frequency:** normally every 4 hours
- **Calendar spacing:** can contain longer gaps


## 11. Count Normal H4 Intervals and Larger Gaps

In [ ]:
normal_interval = pd.Timedelta(hours=4)

normal_count = (time_differences == normal_interval).sum()
larger_gap_count = (time_differences > normal_interval).sum()

print("Normal 4-hour intervals :", normal_count)
print("Intervals larger than 4h:", larger_gap_count)

## 12. Inspect Larger Gaps

In [ ]:
gaps = time_differences[time_differences > pd.Timedelta(hours=4)]
gaps.head(20)

We do **not** automatically fill these gaps. Many are legitimate non-trading periods such as weekends and holidays.


# 13. Select Data by Date and Time

In [ ]:
# One year
eurusd.loc["2025"].head()

In [ ]:
# One month
eurusd.loc["2025-01"].head()

In [ ]:
# Date range
eurusd.loc["2025-01-01":"2025-01-15"].head(20)

In [ ]:
# One trading day
eurusd.loc["2025-01-02"]

For a complete day:

$$
\frac{24}{4} = 6
$$

So we often expect approximately 6 H4 candles, though holidays and broker session conventions can change this.


## 14. Extract Time Components from the Index

In [ ]:
print("Years :", eurusd.index.year[:10])
print("Months:", eurusd.index.month[:10])
print("Hours :", eurusd.index.hour[:10])

### Unique Candle Hours

In [ ]:
sorted(eurusd.index.hour.unique())

Typical H4 candle hours may look like:

`[0, 4, 8, 12, 16, 20]`

This is another indication that candles normally begin every four hours.


## 15. Count Candles per Calendar Day

In [ ]:
candles_per_day = eurusd.groupby(eurusd.index.date).size()
candles_per_day.head(10)

In [ ]:
candles_per_day.value_counts().sort_index()

A normal full day often contains around 6 H4 candles:

$$
24 \div 4 = 6
$$


## 16. Visualize a Small H4 Sample

In [ ]:
sample = eurusd.iloc[:60]

plt.figure(figsize=(14, 6))
plt.plot(sample.index, sample["Close"], marker="o")
plt.xlabel("DateTime")
plt.ylabel("EUR/USD Close")
plt.title("EUR/USD H4 Close Price — First 60 Observations")
plt.xticks(rotation=45)
plt.show()

# 17. Time Index vs Frequency

### Time Index
The actual timestamp of every observation.

Example:
- 2009-12-21 00:00
- 2009-12-21 04:00
- 2009-12-21 08:00

### Frequency
How often observations normally occur.

For this dataset:

**Normal frequency = 4 hours**

So:
- `DateTimeIndex` = when each observation occurred
- H4 = normal sampling frequency


# 18. Do Not Force Calendar H4 Frequency Yet

We will not do:

```python
eurusd.asfreq("4h")
```

because that could create artificial timestamps during weekends and market closures.

We preserve the original trading observations.


# Lesson 2 Practice Questions

1. What type of index does the final DataFrame use?
2. What is the normal sampling interval?
3. Why can `pd.infer_freq()` return `None`?
4. Why do we see gaps longer than 4 hours?
5. Approximately how many H4 candles fit in 24 hours?
6. What is the difference between time index and frequency?
7. Should we automatically fill weekend gaps? Why?


# Suggested Answers

1. `DatetimeIndex`
2. 4 hours
3. Forex has weekend/holiday closures, so calendar spacing is not perfectly regular.
4. Mostly because the market is closed during some periods; a few gaps can also come from the data provider.
5. $24 \div 4 = 6$ candles.
6. The index stores timestamps; frequency describes normal spacing between observations.
7. No. Filling them automatically could create artificial market data.


# Final Practice Task

Using `eurusd`:

1. Print the first and last timestamps.
2. Check whether the index is sorted.
3. Check for duplicate timestamps.
4. Calculate consecutive timestamp differences.
5. Find the most common interval.
6. Confirm whether the data is H4.
7. Display one month of data.
8. Display one trading day.
9. Count candles in that day.
10. Print the unique candle hours.
11. Plot 50 consecutive `Close` observations.


In [ ]:
# Write your Lesson 2 practice code here


# Lesson 2 Summary

You have applied **Time Index, Frequency, and Sampling** to the real EUR/USD dataset.

Key findings:
- The data uses a `DateTimeIndex`.
- Observations are kept in chronological order.
- The dominant interval is 4 hours.
- `EURUSD240.csv` is H4 data.
- Forex contains natural weekend/holiday gaps.
- Frequency inference can fail because calendar intervals are irregular.
- Timestamp differences reveal the underlying trading interval.
- Date-based selection becomes easy with a `DateTimeIndex`.
- A full 24-hour period contains about 6 H4 intervals.
- We keep the original H4 observations unchanged.
